# Gold Layer — Analytics Aggregations

Gold views:
- **daily_sales_fact**: Daily revenue, order count, avg order value
- **customer_metrics**: Lifetime value, total spend, days since last purchase
- **product_metrics**: Total revenue, units sold, avg price by product
- **category_trends**: 7-day moving average of revenue by category

In [ ]:
from pyspark.sql import SparkSession
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = SparkSession.builder \
    .appName("explore_gold") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

In [ ]:
BASE = "/data/delta/gold"
tables = ["daily_sales_fact", "customer_metrics", "product_metrics", "category_trends"]
for t in tables:
    df = spark.read.format("delta").load(f"{BASE}/{t}")
    print(f"\n{'='*60}")
    print(f"{t.upper()}: {df.count()} rows \u00d7 {len(df.columns)} cols")
    df.printSchema()
    df.show(5, truncate=False)

In [ ]:
# Top 10 customers by lifetime value
df = spark.read.format("delta").load(f"{BASE}/customer_metrics")
print("Top 10 customers by lifetime value:")
df.orderBy("lifetime_value", ascending=False).select(
    "customer_id", "total_transactions", "total_spend", "lifetime_value", "days_since_last_purchase"
).show(10, truncate=False)

In [ ]:
# Top products by revenue
df = spark.read.format("delta").load(f"{BASE}/product_metrics")
print("Top 10 products by revenue:")
df.orderBy("total_revenue", ascending=False).select(
    "product_id", "product_name", "category", "total_units_sold", "total_revenue", "avg_price"
).show(10, truncate=False)

In [ ]:
# Daily revenue trend
df = spark.read.format("delta").load(f"{BASE}/daily_sales_fact")
print("Daily sales trend:")
df.orderBy("sale_date").select(
    "sale_date", "total_revenue", "order_count", "avg_order_value"
).show(15, truncate=False)

In [ ]:
# Category trends with 7-day moving average
df = spark.read.format("delta").load(f"{BASE}/category_trends")
print("Category revenue trends (7-day moving avg):")
df.orderBy("category", "trend_date").show(20, truncate=False)

In [ ]:
spark.stop()